# Phishing Email Detection Using Machine Learning and Deep Learning

Phishing emails are a major cybersecurity threat. The objective of this experiment is to build classification models capable of identifying suspicious emails using textual features.

The experiment follows two approaches:

1. Traditional machine learning using TF-IDF features:
   - Logistic Regression
   - Random Forest

2. Deep learning using PyTorch:
   - Long Short-Term Memory (LSTM)

The email text is preprocessed by removing punctuation and stopwords and applying lemmatization. Class imbalance is handled only within the training dataset to prevent information leakage.

The models are evaluated using Precision, Recall, F1-score and ROC-AUC.

In [0]:
# ============================================================
# IMPORT REQUIRED LIBRARIES
# ============================================================

# Data manipulation
import pandas as pd
import numpy as np
import kagglehub
# Text processing
import re
from collections import Counter

# Visualisation
import matplotlib.pyplot as plt

# NLTK
import nltk
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer

# Dataset splitting and resampling
from sklearn.model_selection import train_test_split
from sklearn.utils import resample

# TF-IDF
from sklearn.feature_extraction.text import TfidfVectorizer

# Machine learning models
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier

# Evaluation
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    classification_report,
    confusion_matrix,
    ConfusionMatrixDisplay,
    roc_curve,
    roc_auc_score
)

# PyTorch
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

# Other utilities
import random
import warnings

warnings.filterwarnings("ignore")


# ============================================================
# REPRODUCIBILITY
# ============================================================

SEED = 42

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)


DEVICE = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

print("PyTorch version:", torch.__version__)
print("Device:", DEVICE)

In [0]:
# ============================================================
# DOWNLOAD NLTK RESOURCES
# ============================================================

nltk.download("stopwords")
nltk.download("wordnet")
nltk.download("omw-1.4")

## Dataset Loading

The dataset contains the email subject, email message, class label and date.

The subject and message are combined because both may contain useful information for distinguishing legitimate emails from spam or phishing emails.

In [0]:
# ============================================================
# LOAD DATASET
# ============================================================

path = kagglehub.dataset_download("marcelwiechmann/enron-spam-data")
df = pd.read_csv(path+"/enron_spam_data.csv")


# Display first five records
display(df.head())


print("Dataset shape:", df.shape)

print("\nColumns:")
print(df.columns.tolist())

print("\nDataset information:")
df.info()

In [0]:
# ============================================================
# REMOVE UNNECESSARY INDEX COLUMN
# ============================================================

# This column is usually created when a DataFrame index
# is saved into the CSV file.
df.drop(
    columns=["Unnamed: 0"],
    errors="ignore",
    inplace=True
)


print(df.columns.tolist())

## Data Quality Checking

The dataset is checked for missing values and duplicate records before preprocessing.

Missing subject or message values are replaced with empty strings.

In [0]:
# ============================================================
# CHECK DATA QUALITY
# ============================================================

print("Missing values:\n")
print(df.isnull().sum())


print("\nDuplicate rows:")
print(df.duplicated().sum())